<a href="https://colab.research.google.com/github/nevchia5/dsai-module-2-project/blob/main/netflix_businesscase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Executive Summary

We, The Ai PPL, an AI startup of DSAI Team 4, propose for Netflix to grow revenue from 29.7B in 2021 to 32.6% in 2022 for a 10% increase in revenue through movie-making and movie accumulation in strategic directions.
This will boost viewership and memberships with in-trend offerings that cater to a global audience.

# The Problem Statement
With a growing war chest, Netflix needs a way to reinvest profits into movie-making and procurement in order to stay entrenched as the undisputed global market leader in the subscription video-on-demand (SVOD) streaming industry.


# The Proposed Solution

Trawling through IMDb movie databases and Netflix's own catalog to chart trends for insights, to guide corporate decisions in movie-making/purchasing investment.

# Key Business Insights
* thanks to Daniel's many questions

1. Top genre insights
    *   Which genres contribute 70% of revenue?

2. Top 10 titles and new genre insights
    *   Top genres in Top 10 titles to continue investing in: Which genres are trending with the top 10 titles?
    *   Potential genres for new investment opportunities: Which top 10 titles are contributing to bottom 30% revenue?

3. Make or Buy Investment Insights
    *   Which top 10 titles are purchased from competitors? Can we invest and produce Netflix titles in these genres?



# Costs

type here

# Benefits

type here

# Project Brief

1.   Ingestion of source data into data warehouse
      * Write python scrips to ingest the CSV into the data warehouse

2.   Design star schema
      A. Create dimension and fact tables
      B. Implement schema

3.   ELT Pipeline

4.   Testing Data Quality

5.   Data Analysis using Python showing insights
      A.   Connect to DW using SQLAlchemy
      B.   Pandas: Perform simple analysis
        B1   Top 3 Genre
        B2   Top 10 highly rated Movies
        
6.   Pipeline Orchestration (Optional)

7.   Documentation in Github, Jupyter notebook and slide deck to include
      1.   Code
      2.   Lineage
      3.   Illustrate Pipeline Architecture (eg. DRAW.IO)
      4.   Report summary of technical approach and insights/findings
           with tables, charts and graphs
      5.   Tool choice justification
      6.   Schema choice justification in relation to querying efficiency

8.   Executive stakeholder presentation




















Execution Plan
1. Set up conda environment.

In [ ]:
# Create a new conda environment with Python 3.10
conda create -n bq_elt_pipeline python=3.10 -y

# Activate the environment
conda activate bq_elt_pipeline

# Install the Google Cloud BigQuery client and pandas ecosystem
pip install google-cloud-bigquery pandas pyarrow

VS Code Tip: After creating the environment, open your Python script file in VS Code, click on the Python interpreter version in the bottom-right corner (or press Ctrl+Shift+P / Cmd+Shift+P and type Python: Select Interpreter), and choose bq_elt_pipeline.

2. ELT's 'E' & 'L' phase
Need GCP service account JSON key.


In [ ]:
Bash
export GOOGLE_APPLICATION_CREDENTIALS="/path/to/your/service_account_key.json"

Python script  - load_to_bq_py

In [ ]:
import glob
import os
import pandas as pd
from google.cloud import bigquery

# Initialize the BigQuery client
client = bigquery.Client()

# Define your target BigQuery location (Format: project_id.dataset_id)
# Note: Ensure you manually create the 'staging' and 'final' datasets in BQ Console first
STAGING_DATASET = "your_project_id.staging"

# 1. Load the primary dataset
df1 = pd.read_csv("dataset1.csv")
table_id_1 = f"{STAGING_DATASET}.dataset1_raw"
# Load into BigQuery (Creates table if it doesn't exist, replaces if it does)
client.load_table_from_dataframe(df1, table_id_1).result()
print(f"Successfully loaded primary dataset to {table_id_1}")

# 2. Extract and Load multiple genre CSV files
genre_files = glob.glob("./genre_datasets/*.csv")
table_id_2 = f"{STAGING_DATASET}.dataset2_raw"

# Configure BQ to append data since we are iterating through multiple files
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")

for file_path in genre_files:
    # Extract the genre name from the file name (e.g., 'action.csv' -> 'action')
    genre_name = os.path.basename(file_path).replace(".csv", "")

    df_genre = pd.read_csv(file_path)

    # Clean up column names to prevent BigQuery parsing errors (spaces/weird characters)
    df_genre.columns = df_genre.columns.str.strip().str.replace(' ', '_')

    if 'movie_name' in df_genre.columns:
        # Explicitly tag the file's genre if it's not a column inside the file
        if 'genre' not in df_genre.columns:
            df_genre['genre'] = genre_name

        # Append this slice into the master raw staging table
        client.load_table_from_dataframe(df_genre, table_id_2, job_config=job_config).result()
        print(f"Appended {file_path} to {table_id_2}")

3. ELT's 'T' Phase
Raw data now in BigQuery, we use SQL to
A. Normalize patterns,
B. Deduplicate instances when a movie belongs to multiple genres,
C. Title cleaning

3A. Reusable Normalization User Defined  Function
This UDF in SQL strips punctuation, handle capitalization, spacing and map Roman numerals to standard numeric digits.


SQL

In [ ]:
CREATE OR REPLACE FUNCTION `your_project_id.staging.normalize_title`(title STRING)
RETURNS STRING AS (
  REGEXP_REPLACE(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            LOWER(TRIM(REGEXP_REPLACE(title, r'\s+', ' '))),
          r'\bii\b', '2'),
        r'\biii\b', '3'),
      r'\biv\b', '4'),
    r'\bv\b', '5'),
  r'[^a-z0-9 ]', '')
);

3. Remove Movie 'title' string variance
BigQuery's native EDIT_DISTANCE(a, b) function measures string variance. We normalize this distance against string length to build a comparative index ratio (where 1.0 is an absolute identical match).

To ensure optimal BigQuery query performance and prevent massive cost draw on cross-joins, we utilize a Blocking Prefix constraint (e.g., forcing movies to match on their first letter).

SQL

In [ ]:
CREATE OR REPLACE TABLE `your_project_id.final.enriched_dataset1` AS
WITH prepared_data AS (
  SELECT
    d1.*,
    `your_project_id.staging.normalize_title`(d1.title) AS clean_title
  FROM
    `your_project_id.staging.dataset1_raw` d1
),
calculated_matches AS (
  SELECT
    p.*,
    d2.movie_name AS matched_movie_name,
    d2.genre,
    d2.rating,
    -- Calculate normalized similarity score
    1.0 - (EDIT_DISTANCE(p.clean_title, d2.clean_movie_name) / GREATEST(LENGTH(p.clean_title), 1)) AS similarity_ratio,
    -- Rank matches to capture only the absolute closest record
    ROW_NUMBER() OVER(PARTITION BY p.clean_title ORDER BY 1.0 - (EDIT_DISTANCE(p.clean_title, d2.clean_movie_name) / GREATEST(LENGTH(p.clean_title), 1)) DESC) as match_rank
  FROM
    prepared_data p
  INNER JOIN
    `your_project_id.staging.dataset2_deduped` d2
    -- Performance Block Optimization: Only evaluate pairs starting with the same character
    ON SUBSTR(p.clean_title, 1, 1) = SUBSTR(d2.clean_movie_name, 1, 1)
  WHERE
    ABS(LENGTH(p.clean_title) - LENGTH(d2.clean_movie_name)) <= 6
)
SELECT
  * EXCEPT(clean_title, similarity_ratio, match_rank)
FROM
  calculated_matches
-- Only pick the #1 top match, given it crosses an 80% similarity threshold
WHERE match_rank = 1 AND similarity_ratio >= 0.80;

4. Verify the results
Table is ready for querying or extraction to CSV from VS Code using SQL:

In [ ]:
SELECT title, matched_movie_name, genre, rating
FROM `your_project_id.final.enriched_dataset1`
LIMIT 100;

Source Data
IMDb
netflix

# TEAM,
Still not complete yet.
Needs IMDb dataset to match viewer ratings and genre.